# Functions and interfaces through probability theory

Represent probability mass functions as Python callables, compose them into expectations and event
probabilities, and connect mathematical contracts to machine-learning code.

**Lecture 2 · Python Foundations II · CMOR 438 / INDE 577**


## Python focus: probability is the teaching context

This is a **Python functions lesson**, not a survey of probability theory. The notebook states
the small amount of probability it needs, and the probability objects give each Python feature
a meaningful job:

| Python learning target | Probability object used to teach it |
| --- | --- |
| function definition and return value | probability mass function |
| parameters, keywords, and validation | Bernoulli parameter `p` |
| functions as ordinary objects | events and random variables |
| higher-order functions | event probability and expectation |
| closures | a family of Bernoulli distributions |
| explicit state and reproducibility | seeded simulation |
| boundary validation | probability mass and binary log loss |

Students should be able to explain the Python call, scope, returned object, and failure behavior.
Any probability identity used in an exercise is introduced before it is required.


## How to use this notebook

**Estimated time:** 50 minutes of core instruction, plus 30–40 minutes of practice and extension.

**Prerequisite:** Lecture 1's values, collections, iteration, comprehensions, and generators.

Follow **Core** during class. Before each function call, identify the arguments, expected return
value, and possible contract violation. **Practice** cells contain executable mathematical checks.
**Extension** sections deepen exact arithmetic, simulation, and machine-learning loss.

We use finite probability spaces so every result can be inspected by hand. The goal is to learn
function design and probabilistic reasoning—not to replace a probability course or library.


## Learning objectives

By the end of this notebook, you should be able to:

- translate a mathematical map into a Python function contract;
- distinguish parameters, arguments, local names, return values, and side effects;
- design positional and keyword-only parameters that make calls readable;
- explain default-argument evaluation and the mutable-default trap;
- combine docstrings, type hints, validation, and exceptions;
- pass functions as objects to compute event probabilities and expectations;
- use a closure to construct a parameterized probability mass function;
- make randomness reproducible by passing an explicit generator;
- check nonnegativity and total-mass axioms with justified tolerances; and
- connect Bernoulli probability to binary log loss without treating every score as calibrated.


## Why this matters in industry

Machine learning repeatedly composes functions: preprocessing maps, model predictors, probability
links, losses, metrics, optimizers, and decision rules. Weak interfaces hide global state, mutate
caller-owned data, accept ambiguous values, or print when downstream code needs a result.

Probability adds semantic risks. A value in `[0, 1]` is not automatically a probability. A vector
of nonnegative scores is not a distribution unless its mass is normalized. A reproducible simulation
requires controlled random state. Clipping a predicted probability changes the loss and must be an
explicit numerical policy.

Professional code makes the mathematical domain and computational behavior part of one testable
contract.


## Mathematical question and running example

Let $X$ be a Bernoulli random variable on the finite sample space containing the outcomes
**failure** and **success**. For $p$ in `[0, 1]`,

$$
P(X=	ext{success})=p,    P(X=	ext{failure})=1-p.
$$

Its expectation and variance are

$$
E[X]=p,    Var(X)=p(1-p),
$$

when success is encoded as one and failure as zero.

Our question is: **How should probability models and random variables be represented so their
contracts can be composed, inspected, and tested?**


## Professional practice: mathematical and software contracts

| Probabilist or data scientist asks | Software engineer asks |
| --- | --- |
| What is the sample space? | Which inputs are accepted or rejected? |
| Is mass nonnegative and normalized? | Where are those invariants checked? |
| What random variable is being averaged? | What callable signature represents it? |
| Is a model output calibrated? | Is it named `score` or `probability` accurately? |
| Which randomness defines the experiment? | Is random state explicit and reproducible? |

A type hint saying `float` cannot establish that a number lies in `[0, 1]`, and a passing assertion
cannot establish calibration on a population. Different claims require different evidence.


## Core: `def` creates a function object

A definition gives a function a name, parameters, a body, and usually an explicit `return`. Defining
the function does not execute the body. A call binds arguments to parameters in a new local scope,
executes the body, and produces the returned object.

The function below represents a Bernoulli probability mass function. Its finite software contract
rejects outcomes outside the declared sample space rather than silently treating a typo as mass zero.


In [ ]:
def bernoulli_pmf(
    outcome: str,
    *,
    success_probability: float,
) -> float:
    """Evaluate a Bernoulli probability mass function.

    Parameters
    ----------
    outcome : {'failure', 'success'}
        Outcome whose probability mass is requested.
    success_probability : float
        Probability of success on the closed interval `[0, 1]`.

    Returns
    -------
    float
        Probability mass assigned to `outcome`.

    Raises
    ------
    TypeError
        If `outcome` is not a string or `success_probability` is not a
        real-valued integer or float.
    ValueError
        If the probability is outside `[0, 1]` or `outcome` is unknown.

    Examples
    --------
    >>> bernoulli_pmf("success", success_probability=0.7)
    0.7
    >>> bernoulli_pmf("failure", success_probability=0.7)
    0.30000000000000004
    """

    if not isinstance(outcome, str):
        raise TypeError(
            "outcome must be a string; "
            f"got {type(outcome).__name__}"
        )
    if isinstance(success_probability, bool) or not isinstance(
        success_probability,
        (int, float),
    ):
        raise TypeError(
            "success_probability must be an int or float; "
            f"got {type(success_probability).__name__}"
        )
    if not 0.0 <= success_probability <= 1.0:
        raise ValueError(
            "success_probability must lie in [0, 1]; "
            f"got {success_probability!r}"
        )
    if outcome == "success":
        return float(success_probability)
    if outcome == "failure":
        return 1.0 - float(success_probability)
    raise ValueError(
        "outcome must be 'failure' or 'success'; "
        f"got {outcome!r}"
    )


assert bernoulli_pmf("success", success_probability=0.7) == 0.7
assert abs(bernoulli_pmf("failure", success_probability=0.7) - 0.3) < 1e-12
assert callable(bernoulli_pmf)


## Core: parameters define the interface; arguments supply one call

`outcome` and `success_probability` are parameters. The string and number in a call are arguments.
The bare `*` makes `success_probability` keyword-only, so the call exposes which probability the
number represents.

Positional arguments are concise for an obvious primary object. Keywords improve clarity for
configuration, Boolean switches, units, and numbers that could be swapped without a type error.


In [ ]:
success_mass = bernoulli_pmf("success", success_probability=0.25)
failure_mass = bernoulli_pmf("failure", success_probability=0.25)

assert success_mass == 0.25
assert failure_mass == 0.75

try:
    bernoulli_pmf("success", 0.25)
except TypeError as error:
    print(f"Captured {type(error).__name__}: {error}")

try:
    bernoulli_pmf("sucess", success_probability=0.25)
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: return data; print only at a presentation boundary

`return value` ends the call and gives data to the caller. Reaching the end without `return` produces
`None`. Printing is a side effect and cannot be composed into an expectation or probability check.

Notebook cells may print returned values for inspection. Reusable mathematical functions should
return results and let another layer decide how to display or log them.


In [ ]:
def print_success_mass(success_probability: float) -> None:
    print(success_probability)


printed_result = print_success_mass(0.7)
returned_result = bernoulli_pmf("success", success_probability=0.7)

assert printed_result is None
assert returned_result == 0.7
assert 2.0 * returned_result == 1.4


## Core: documentation, types, and validation do different jobs

A useful contract states the meaning and domain of every input, the returned object, possible
exceptions, mutation behavior, and mathematical guarantees.

- A **docstring** explains the contract to people and documentation tools.
- **Type hints** help readers, editors, and static analyzers.
- **runtime validation** checks properties such as interval membership.
- **tests** record selected examples and invariants.

No one mechanism replaces the others.


In [ ]:
assert bernoulli_pmf.__name__ == "bernoulli_pmf"
assert "Bernoulli probability mass" in bernoulli_pmf.__doc__
assert bernoulli_pmf.__annotations__["outcome"] is str
assert bernoulli_pmf.__annotations__["return"] is float

try:
    bernoulli_pmf("success", success_probability=1.2)
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Core: docstrings are part of the public interface

A comment helps a maintainer understand an implementation. A **docstring** is attached to a module,
class, method, or function and helps a caller use its public behavior. Python exposes it through
`object.__doc__`, `help`, IDE hover text, and documentation generators.

PEP 257 establishes baseline conventions: use triple double quotes, begin with an imperative summary
such as “Compute” or “Return,” separate a multiline summary from its detail, and document parameters,
returns, side effects, raised exceptions, and restrictions when relevant.

Documentation is executable project infrastructure only when teams review and validate it. A stale
docstring is worse than a short accurate one because users reasonably treat it as a contract.


## Core: docstring styles standardize navigation and tooling

Python does not enforce one multiline layout. Here an **API** (application programming interface) is
the supported contract other code uses—such as a function's signature, behavior, return value, and
exceptions. Documentation generators can publish pages describing that interface. Common docstring
conventions include:

| Style | Recognizable structure | Typical ecosystem |
| --- | --- | --- |
| NumPy/numpydoc | `Parameters` with underlines, `Returns`, `Raises`, `Notes`, `Examples` | scientific Python |
| Google | `Args:`, `Returns:`, `Raises:` blocks | many application projects |
| reStructuredText/Sphinx | field lists such as `:param name:` and `:return:` | Sphinx-based APIs |

The styles describe the same kinds of contracts. Consistency matters because readers learn where to
look, IDEs can display predictable content, and documentation tools can render structured API pages.

**Course convention:** reusable `rice_dsm` APIs use **NumPy style**. We choose it because students will
encounter it throughout NumPy, SciPy, pandas, and the scientific Python ecosystem—not because other
styles are incorrect.


## Core: anatomy of a NumPy-style function docstring

Use only sections that add information:

1. **Summary** — one imperative line stating observable behavior.
2. **Extended summary** — important semantics, assumptions, or algorithmic context.
3. **Parameters** — meaning, domain, units, shape, and optional/default behavior.
4. **Returns** or **Yields** — returned representation and semantics.
5. **Raises** — deliberate, caller-relevant failures and their conditions.
6. **Notes** — mathematical or implementation details needed for correct use.
7. **Examples** — small doctest-style usage that remains readable in plain text.

Do not restate only the signature. Type hints already say `float`; the docstring must say “probability
of success on `[0, 1]`.” Document units, ordering, calibration assumptions, mutation, and numerical
policy—the meaning a type system cannot express.


In [ ]:
import inspect

bernoulli_documentation = inspect.getdoc(bernoulli_pmf)

assert bernoulli_documentation is not None
assert bernoulli_documentation.startswith("Evaluate a Bernoulli")
assert "Parameters\n----------" in bernoulli_documentation
assert "Returns\n-------" in bernoulli_documentation
assert "Raises\n------" in bernoulli_documentation
assert "Examples\n--------" in bernoulli_documentation

print(bernoulli_documentation)


## Core: type hints and docstrings are complementary

Annotations form a machine-readable interface for editors, refactoring tools, and static type
checkers. They are not runtime guards. Use precise abstractions: `Iterable[str]` promises any
reusable or one-pass iterable, while `list[str]` requires list behavior; `Callable[[str], float]`
states a callable's argument and return structure.

Avoid `Any` merely to silence a checker. If inputs are genuinely broad, define the behavior they must
support. At runtime, validate domain properties types cannot express, such as probability intervals,
nonempty unique outcomes, finiteness, and total mass.

The course will run a static checker when these interfaces move into the package. Notebook assertions
exercise runtime behavior; the two checks find different defects.


In [ ]:
from typing import get_type_hints

bernoulli_hints = get_type_hints(bernoulli_pmf)
assert bernoulli_hints == {
    "outcome": str,
    "success_probability": float,
    "return": float,
}
# Annotations are metadata: the function still needs an intentional runtime error.
try:
    bernoulli_pmf("success", success_probability="0.7")
except TypeError as error:
    assert "success_probability" in str(error)
    assert "str" in str(error)
    print(f"Captured {type(error).__name__}: {error}")


## Core: exceptions are part of a usable programming interface (API)

API stands for **application programming interface**: the documented contract through which other
code uses a function, class, package, or service. For a Python function, its API includes its import
path, signature, parameter meanings, return behavior, exceptions, and promised side effects. The
function body is usually an implementation detail. A web API applies the same boundary idea between
processes using requests and responses; an API does not inherently require the internet.

Raise an exception when a function cannot honor its return contract:

- `TypeError` for an unsupported kind of object;
- `ValueError` for the right general type with an invalid domain value;
- `KeyError` when a required mapping key is absent;
- a custom `Exception` subclass when callers need one stable domain-specific failure category.

An actionable message names the parameter, expected domain, and safe representation of the received
value. Avoid secrets or entire records in production errors. Library code should generally raise,
not print an error or return an ambiguous `None`. Catch errors at a layer that can actually recover,
add context with exception chaining, and never use a broad `except Exception` to hide defects.


In [ ]:
def parse_probability(text: str) -> float:
    """Parse decimal text as a probability.

    Parameters
    ----------
    text : str
        Decimal representation of a probability on `[0, 1]`.

    Returns
    -------
    float
        Parsed probability.

    Raises
    ------
    TypeError
        If `text` is not a string.
    ValueError
        If `text` is not decimal text or the parsed value is outside `[0, 1]`.

    Examples
    --------
    >>> parse_probability("0.875")
    0.875
    """

    if not isinstance(text, str):
        raise TypeError(f"text must be a string; got {type(text).__name__}")
    try:
        probability = float(text)
    except ValueError as error:
        raise ValueError(
            f"text must contain a decimal probability; got {text!r}"
        ) from error
    if not 0.0 <= probability <= 1.0:
        raise ValueError(
            "parsed probability must lie in [0, 1]; "
            f"got {probability!r} from text={text!r}"
        )
    return probability


assert parse_probability("0.875") == 0.875

try:
    parse_probability("seventy percent")
except ValueError as error:
    assert isinstance(error.__cause__, ValueError)
    assert "decimal probability" in str(error)
    print(f"Captured {type(error).__name__}: {error}")


## Professional practice: examples, tests, and static checks form a system

- **Docstring examples** show the happy path in the caller's vocabulary.
- **Unit tests** cover boundaries, invalid inputs, invariants, and regressions.
- **Static type checks** compare annotated interfaces without running the program.
- **Runtime validation** protects assumptions at real input boundaries.
- **CI** runs all of them on every proposed change.

Examples are not a replacement for tests, and a `Raises` section is not a replacement for actually
raising the documented exception. Industry-grade documentation stays aligned with observable behavior
through review and automated checks.


## Core: default arguments are evaluated when `def` executes

Defaults are created once at definition time, not once per call. An immutable mathematical constant
may be reasonable when it is genuinely part of the interface. A mutable default such as `history=[]`
creates hidden shared state across calls.

Use `None` as a sentinel when each call should create a new mutable object. Do not hide a substantive
model assumption—such as class prevalence—inside an unexplained default.


In [ ]:
def bad_record_probability(
    probability: float,
    history: list[float] = [],
) -> list[float]:
    """Demonstrate shared default state; do not copy this design."""

    history.append(probability)
    return history


first_history = bad_record_probability(0.2)
second_history = bad_record_probability(0.8)

assert first_history is second_history
assert second_history == [0.2, 0.8]


def record_probability(
    probability: float,
    history: list[float] | None = None,
) -> list[float]:
    if history is None:
        history = []
    history.append(probability)
    return history


assert record_probability(0.2) == [0.2]
assert record_probability(0.8) == [0.8]


## Core: explicit inputs beat hidden global state

Python resolves names through local, enclosing, global, and built-in scopes. Parameters and names
assigned inside a function are local unless declared otherwise.

Probability code is easier to test and reproduce when distributions, thresholds, and random state
arrive as explicit inputs. Avoid functions whose result changes because a notebook cell silently
reassigned a global variable.


In [ ]:
success_probability = 0.99


def local_mass(success_probability: float) -> float:
    result = success_probability
    return result


assert local_mass(0.4) == 0.4
assert success_probability == 0.99


## Core: functions are ordinary objects

A probability mass function, event indicator, or random variable can be represented by a callable.
We can store it in a name, pass it to another function, and return it from a function.

A function accepting or returning another function is **higher-order**. This lets one expectation
routine operate on many distributions and random variables without embedding their definitions.


In [ ]:
selected_mass_function = bernoulli_pmf

assert selected_mass_function is bernoulli_pmf
assert callable(selected_mass_function)
assert selected_mass_function("success", success_probability=0.6) == 0.6


## Core: a closure can bind a distribution parameter

The general `bernoulli_pmf` takes both an outcome and `success_probability`. A consumer such as an
expectation routine is clearer if it receives a one-argument mass function.

A nested function can retain names from its enclosing scope. `make_bernoulli_pmf` validates `p` once
and returns a closure with signature `(outcome) -> mass`, avoiding mutable global state.


In [ ]:
from collections.abc import Callable, Iterable


def make_bernoulli_pmf(
    success_probability: float,
) -> Callable[[str], float]:
    """Construct a Bernoulli probability mass function.

    Parameters
    ----------
    success_probability : float
        Probability of success on the closed interval `[0, 1]`.

    Returns
    -------
    Callable[[str], float]
        One-argument function mapping an outcome to its probability mass.

    Raises
    ------
    TypeError
        If `success_probability` is not a real-valued integer or float.
    ValueError
        If `success_probability` lies outside `[0, 1]`.

    Notes
    -----
    The returned closure retains `success_probability` without reading mutable
    global state.

    Examples
    --------
    >>> pmf = make_bernoulli_pmf(0.25)
    >>> pmf("success")
    0.25
    """

    # Reuse the public boundary's type and interval validation once.
    bernoulli_pmf("success", success_probability=success_probability)

    def pmf(outcome: str) -> float:
        return bernoulli_pmf(
            outcome,
            success_probability=success_probability,
        )

    return pmf


fair_pmf = make_bernoulli_pmf(0.5)
biased_pmf = make_bernoulli_pmf(0.7)

assert fair_pmf("success") == 0.5
assert biased_pmf("success") == 0.7
assert fair_pmf is not biased_pmf


## Core: event probability and expectation are higher-order operations

For finite outcomes, an event is a predicate and a random variable is a numerical function:

$$
P(A)=Σ_{ω ∈ Ω} 1_A(ω)P(ω),
$$

$$
E[g(X)]=Σ_{ω ∈ Ω} g(ω)P(ω).
$$

The implementations accept behavior as function objects. They return values without knowing the
specific distribution, event, or random variable.


In [ ]:
def event_probability(
    outcomes: Iterable[str],
    pmf: Callable[[str], float],
    event: Callable[[str], bool],
) -> float:
    """Compute the probability of an event on a finite sample space.

    Parameters
    ----------
    outcomes : Iterable[str]
        Complete finite sample space. Outcomes must appear exactly once.
    pmf : Callable[[str], float]
        Function assigning probability mass to each outcome.
    event : Callable[[str], bool]
        Predicate returning whether an outcome belongs to the event.

    Returns
    -------
    float
        Sum of probability masses for outcomes in the event.

    Notes
    -----
    This function assumes `pmf` is already validated. It does not normalize
    masses or detect duplicate outcomes.
    """

    return sum(pmf(outcome) for outcome in outcomes if event(outcome))


def expected_value(
    outcomes: Iterable[str],
    pmf: Callable[[str], float],
    random_variable: Callable[[str], float],
) -> float:
    """Compute the expectation of a finite real random variable.

    Parameters
    ----------
    outcomes : Iterable[str]
        Complete finite sample space. Outcomes must appear exactly once.
    pmf : Callable[[str], float]
        Validated probability mass function on `outcomes`.
    random_variable : Callable[[str], float]
        Function mapping each outcome to a real value.

    Returns
    -------
    float
        Probability-weighted sum of `random_variable` over `outcomes`.

    Notes
    -----
    Duplicate or missing outcomes change the result. Validation of sample-space
    completeness belongs to the caller's domain boundary.
    """

    return sum(random_variable(outcome) * pmf(outcome) for outcome in outcomes)


def is_success(outcome: str) -> bool:
    """Return whether `outcome` is the Bernoulli success outcome."""

    return outcome == "success"


def success_indicator(outcome: str) -> float:
    """Return one for success and zero otherwise."""

    return 1.0 if outcome == "success" else 0.0


bernoulli_outcomes = ("failure", "success")

assert event_probability(bernoulli_outcomes, biased_pmf, is_success) == 0.7
assert expected_value(bernoulli_outcomes, biased_pmf, success_indicator) == 0.7


## Core: validate probability axioms separately from types

On a finite sample space, a PMF must assign nonnegative mass and total mass one. Floating-point
arithmetic may produce a tiny rounding difference, so use a justified tolerance rather than rounding
the distribution until a test passes.

Validation on listed outcomes also assumes the list completely describes the sample space. A valid
sum over an incomplete set is not evidence that the intended distribution is normalized.


In [ ]:
import math


def validate_finite_pmf(
    outcomes: Iterable[str],
    pmf: Callable[[str], float],
    *,
    absolute_tolerance: float = 1e-12,
) -> None:
    """Validate a probability mass function on a finite sample space.

    Parameters
    ----------
    outcomes : Iterable[str]
        Complete finite sample space. Outcomes must be unique.
    pmf : Callable[[str], float]
        Function assigning mass to every supplied outcome.
    absolute_tolerance : float, default=1e-12
        Maximum accepted absolute error in total probability mass.

    Returns
    -------
    None
        The function returns only when every checked invariant holds.

    Raises
    ------
    ValueError
        If the sample space is empty or duplicated, tolerance is invalid, a
        mass is non-finite or negative, or total mass differs from one beyond
        `absolute_tolerance`.

    Examples
    --------
    >>> validate_finite_pmf(("failure", "success"), make_bernoulli_pmf(0.4))
    """

    outcome_tuple = tuple(outcomes)
    if not outcome_tuple:
        raise ValueError("outcomes must define a nonempty sample space")
    if len(set(outcome_tuple)) != len(outcome_tuple):
        raise ValueError(
            "outcomes must be unique; "
            f"got {outcome_tuple!r}"
        )
    if absolute_tolerance < 0.0 or not math.isfinite(absolute_tolerance):
        raise ValueError(
            "absolute_tolerance must be finite and nonnegative; "
            f"got {absolute_tolerance!r}"
        )

    masses = [pmf(outcome) for outcome in outcome_tuple]
    if any(not math.isfinite(mass) or mass < 0.0 for mass in masses):
        raise ValueError(
            "pmf must return finite, nonnegative masses; "
            f"got {masses!r}"
        )

    total_mass = sum(masses)
    if not math.isclose(
        total_mass,
        1.0,
        rel_tol=0.0,
        abs_tol=absolute_tolerance,
    ):
        raise ValueError(
            "probability masses must sum to one within "
            f"absolute_tolerance={absolute_tolerance!r}; "
            f"got total_mass={total_mass!r}"
        )


assert validate_finite_pmf(bernoulli_outcomes, biased_pmf) is None


def invalid_pmf(outcome: str) -> float:
    return {"failure": 0.4, "success": 0.7}[outcome]


try:
    validate_finite_pmf(bernoulli_outcomes, invalid_pmf)
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Worked example: compute Bernoulli moments from reusable functions

We will compute $E[X]$, $E[X^2]$, and $Var(X)=E[X^2]-E[X]^2$ for `p=0.7`:

1. create a validated PMF closure;
2. validate total probability mass;
3. define the numerical random variables;
4. call the general expectation interface; and
5. compare with the analytical formulas.

The distribution, random variable, and aggregation remain separate and independently testable.


In [ ]:
probability_of_success = 0.7
worked_pmf = make_bernoulli_pmf(probability_of_success)
validate_finite_pmf(bernoulli_outcomes, worked_pmf)


def bernoulli_value(outcome: str) -> float:
    return 1.0 if outcome == "success" else 0.0


def squared_bernoulli_value(outcome: str) -> float:
    value = bernoulli_value(outcome)
    return value**2


mean = expected_value(bernoulli_outcomes, worked_pmf, bernoulli_value)
second_moment = expected_value(
    bernoulli_outcomes,
    worked_pmf,
    squared_bernoulli_value,
)
variance = second_moment - mean**2

assert math.isclose(mean, probability_of_success)
assert math.isclose(second_moment, probability_of_success)
assert math.isclose(variance, probability_of_success * (1.0 - probability_of_success))
assert math.isclose(sum(worked_pmf(outcome) for outcome in bernoulli_outcomes), 1.0)

print(f"mean={mean:.3f}, variance={variance:.3f}")


## Core: multiple return values form one tuple

A small function may return related results as one tuple, which the caller can retain or unpack.
This is suitable for a local, fixed grouping such as `(mean, variance)`. When a result gains many
fields, units, or validation rules, the next notebook's dataclass may communicate better.


In [ ]:
def bernoulli_moments(success_probability: float) -> tuple[float, float]:
    """Return analytical (mean, variance) for a Bernoulli random variable."""

    if not 0.0 <= success_probability <= 1.0:
        raise ValueError("success_probability must lie in [0, 1]")
    return (
        success_probability,
        success_probability * (1.0 - success_probability),
    )


moment_pair = bernoulli_moments(0.7)
analytical_mean, analytical_variance = moment_pair

assert moment_pair == (0.7, 0.21000000000000002)
assert analytical_mean == 0.7
assert math.isclose(analytical_variance, 0.21)


## Extension: make random state an explicit dependency

Sampling is intentionally stateful. Calling a pseudorandom generator advances its state, so an
otherwise identical call can return a different outcome. Pass a `random.Random` instance rather than
using hidden module-level state. A fixed seed makes a teaching experiment reproducible; it does not
make samples genuinely random or justify a statistical conclusion.

Simulation checks can expose implementation errors and illustrate convergence. They do not replace
exact calculation when exact calculation is available.


In [ ]:
import random


def sample_bernoulli(
    success_probability: float,
    *,
    generator: random.Random,
) -> str:
    """Draw one reproducible Bernoulli outcome using an explicit generator."""

    if not 0.0 <= success_probability <= 1.0:
        raise ValueError("success_probability must lie in [0, 1]")
    return "success" if generator.random() < success_probability else "failure"


generator = random.Random(2026)
draws = [
    sample_bernoulli(0.7, generator=generator)
    for _ in range(20_000)
]
empirical_probability = draws.count("success") / len(draws)

assert len(draws) == 20_000
assert set(draws) <= {"failure", "success"}
assert abs(empirical_probability - 0.7) < 0.02

print(f"empirical success frequency={empirical_probability:.4f}")


## Extension: Bernoulli probability connects to binary log loss

For label $y$ in `{0, 1}` and predicted probability $q$ strictly between zero and one,

$$
L(y,q)=-[y ln(q) + (1-y) ln(1-q)].
$$

This is the negative log-likelihood of a Bernoulli observation. The function validates its finite
mathematical domain. Production systems often clip extreme probabilities for numerical reasons, but
clipping changes the evaluated prediction and must be documented rather than hidden.


In [ ]:
def binary_log_loss(label: int, predicted_probability: float) -> float:
    """Compute Bernoulli negative log-likelihood for one observation.

    Parameters
    ----------
    label : {0, 1}
        Observed binary class label.
    predicted_probability : float
        Predicted probability of label one, strictly between zero and one.

    Returns
    -------
    float
        Nonnegative negative log-likelihood for the observation.

    Raises
    ------
    TypeError
        If `label` is not an integer or `predicted_probability` is not an
        integer or float.
    ValueError
        If `label` is not zero or one, or probability is outside `(0, 1)`.

    Notes
    -----
    This function does not clip probabilities. Clipping is a separate numerical
    policy that changes the evaluated prediction and must be explicit.

    Examples
    --------
    >>> round(binary_log_loss(1, 0.9), 4)
    0.1054
    """

    if isinstance(label, bool) or not isinstance(label, int):
        raise TypeError(
            "label must be an integer; "
            f"got {type(label).__name__}"
        )
    if isinstance(predicted_probability, bool) or not isinstance(
        predicted_probability,
        (int, float),
    ):
        raise TypeError(
            "predicted_probability must be an int or float; "
            f"got {type(predicted_probability).__name__}"
        )
    if label not in (0, 1):
        raise ValueError(f"label must be 0 or 1; got {label!r}")
    if not 0.0 < predicted_probability < 1.0:
        raise ValueError(
            "predicted_probability must lie strictly between 0 and 1; "
            f"got {predicted_probability!r}"
        )
    if label == 1:
        return -math.log(predicted_probability)
    return -math.log1p(-predicted_probability)


assert binary_log_loss(1, 0.9) < binary_log_loss(1, 0.2)
assert binary_log_loss(0, 0.1) < binary_log_loss(0, 0.8)

try:
    binary_log_loss(1, 1.0)
except ValueError as error:
    print(f"Captured {type(error).__name__}: {error}")


## Debugging playbook for functions and probability code

Inspect in this order:

1. the signature — which arguments are positional or keyword-only?
2. `repr` and `type` of each boundary input;
3. the returned object — number, tuple, callable, or unexpected `None`?
4. sample-space completeness and uniqueness;
5. every mass for finiteness and nonnegativity;
6. total mass with a justified tolerance;
7. mutable defaults, global parameters, and hidden random state;
8. the smallest direct call to the PMF, event, and random-variable functions;
9. whether the claimed probability interpretation requires calibration evidence.

Separate an interface failure from a probability-model failure before changing either layer.


## Practice: guided two-trial probability space

Use two independent trials with `p=0.7`. Define a joint PMF as the product of the two marginal masses,
an event for at least one success, and a random variable counting successes.

Then verify total mass, $P(	ext{at least one success})=0.91$, and expected success count `1.4`.
The assertions are success criteria.


In [ ]:
two_trial_outcomes = (
    ("failure", "failure"),
    ("failure", "success"),
    ("success", "failure"),
    ("success", "success"),
)


def two_trial_pmf(outcome_pair: tuple[str, str]) -> float:
    first, second = outcome_pair
    return worked_pmf(first) * worked_pmf(second)


def at_least_one_success(outcome_pair: tuple[str, str]) -> bool:
    return "success" in outcome_pair


def success_count(outcome_pair: tuple[str, str]) -> float:
    return float(outcome_pair.count("success"))


joint_mass = sum(two_trial_pmf(outcome) for outcome in two_trial_outcomes)
at_least_one_probability = sum(
    two_trial_pmf(outcome)
    for outcome in two_trial_outcomes
    if at_least_one_success(outcome)
)
expected_successes = sum(
    success_count(outcome) * two_trial_pmf(outcome)
    for outcome in two_trial_outcomes
)

assert math.isclose(joint_mass, 1.0)
assert math.isclose(at_least_one_probability, 0.91)
assert math.isclose(expected_successes, 1.4)


## Practice: independent fair-die functions

For outcomes `1` through `6`, define:

- a uniform die PMF;
- an event identifying even outcomes; and
- a random variable returning the square of an outcome.

Compute total mass, the probability of an even result, $E[X]$, and $E[X^2]$. Preserve exact intent
with clear return values and pass the assertions below.


In [ ]:
die_outcomes = tuple(range(1, 7))


def fair_die_pmf(outcome: int) -> float:
    if outcome not in die_outcomes:
        raise ValueError(f"unknown die outcome: {outcome!r}")
    return 1.0 / 6.0


def is_even(outcome: int) -> bool:
    return outcome % 2 == 0


def die_value(outcome: int) -> float:
    return float(outcome)


def squared_die_value(outcome: int) -> float:
    return float(outcome**2)


die_total_mass = sum(fair_die_pmf(outcome) for outcome in die_outcomes)
even_probability = sum(
    fair_die_pmf(outcome) for outcome in die_outcomes if is_even(outcome)
)
die_mean = sum(die_value(outcome) * fair_die_pmf(outcome) for outcome in die_outcomes)
die_second_moment = sum(
    squared_die_value(outcome) * fair_die_pmf(outcome)
    for outcome in die_outcomes
)

assert math.isclose(die_total_mass, 1.0)
assert math.isclose(even_probability, 0.5)
assert math.isclose(die_mean, 3.5)
assert math.isclose(die_second_moment, 91.0 / 6.0)


## Extension: exact arithmetic can clarify finite examples

`fractions.Fraction` represents rational numbers exactly. It is useful for small symbolic or teaching
calculations where exact equality is the intended mathematics. It is not a drop-in performance
replacement for floating-point machine-learning arrays.

Recompute the fair-die total mass and expectation with fractions, then explain why production model
training generally uses floating-point arithmetic and numerical tolerances instead.


In [ ]:
from fractions import Fraction

exact_die_mass = sum((Fraction(1, 6) for _ in die_outcomes), start=Fraction(0, 1))
exact_die_mean = sum(
    (Fraction(outcome, 6) for outcome in die_outcomes),
    start=Fraction(0, 1),
)

assert exact_die_mass == Fraction(1, 1)
assert exact_die_mean == Fraction(7, 2)


## Extension: make and defend a prediction interface

A classifier returns three nonnegative numbers for classes `A`, `B`, and `C`. A teammate names the
output `probabilities` and selects the largest value.

Write an interface proposal addressing:

1. whether the values sum to one and are finite;
2. whether they are logits, scores, normalized masses, or calibrated probabilities;
3. how class order is represented and tested;
4. how ties and abstention are handled;
5. whether normalization belongs inside this function;
6. which calibration evidence is required before using probability language.

Good naming is part of mathematical correctness, not merely style.


## Common failure modes

| Symptom | Likely mistake | Better response |
| --- | --- | --- |
| downstream result is `None` | function printed instead of returned | return data; display separately |
| histories contaminate later calls | mutable default reused | use `None` sentinel or explicit state |
| invalid probability passes hints | annotations do not enforce intervals | validate mathematical domains |
| probability check passes incomplete outcomes | sample space omitted cases | validate completeness separately |
| repeated simulation changes mysteriously | random state is hidden | pass an explicit seeded generator |
| different PMFs require copied code | distribution embedded in aggregation | pass a callable or construct a closure |
| log loss becomes infinite/domain error | prediction equals zero or one | state and test a numerical boundary policy |
| score is called probability | range confused with calibration | use accurate names and calibration evidence |
| exact equality fails for decimal mass | floating-point rounding | use justified tolerance or exact rational arithmetic |

Function mechanics and probability semantics must both be correct.


## Retrieval practice

Answer without running code:

1. When does a function body execute?
2. What is the difference between a parameter and an argument?
3. Why is printing not a substitute for returning?
4. What do type hints communicate but not enforce?
5. Why is `success_probability` keyword-only?
6. When are default arguments evaluated?
7. How does a closure turn a family of Bernoulli PMFs into one callable?
8. Why are expectation and event probability higher-order operations here?
9. What must be checked before nonnegative scores are called a PMF?
10. Why does a fixed random seed support reproducibility but not statistical validity?


## Takeaway and next step

Functions make mathematical contracts composable. Strong probability code has explicit sample
spaces and parameters, composable return values, domain validation, controlled state, and tests for
normalization and interpretation. Higher-order functions separate a distribution, event, random
variable, and aggregation instead of hard-coding them together.

Notebook 01 asks when related state and behavior deserve a named class or dataclass rather than a
larger collection of functions and dictionaries.


## Further reading

- [Python tutorial: defining functions](https://docs.python.org/3.12/tutorial/controlflow.html#defining-functions)
- [Python function definitions](https://docs.python.org/3.12/reference/compound_stmts.html#function-definitions)
- [Python call expressions](https://docs.python.org/3.12/reference/expressions.html#calls)
- [Python typing documentation](https://docs.python.org/3.12/library/typing.html)
- [Python `math.isclose`](https://docs.python.org/3.12/library/math.html#math.isclose)
- [Python `random` module](https://docs.python.org/3.12/library/random.html)
- [Python `fractions.Fraction`](https://docs.python.org/3.12/library/fractions.html)
- [PEP 257: docstring conventions](https://peps.python.org/pep-0257/)
- [numpydoc style guide](https://numpydoc.readthedocs.io/en/v1.10.0/format.html)
- [Python errors and exceptions](https://docs.python.org/3.12/tutorial/errors.html)
- [PEP 3102: keyword-only arguments](https://peps.python.org/pep-3102/)

The Python documentation specifies language behavior. Probability interpretation, calibration, and
statistical guarantees require mathematical and empirical evidence beyond type-correct code.
